# AH | Exploración de adopción digital

**Autor:** Alan Arturo Hernández Gutiérrez.

Se perfilan siete tablas de una muestra de comercio minorista: granularidad, claves, cobertura, fechas, importes y oportunidades analíticas. Las fuentes se leen sin alterarlas. La preparación dimensional está en el segundo cuaderno.

Abra Jupyter desde la raíz del proyecto o `notebooks/`. Se usa el ZIP autorizado de `input/` si existe; también puede indicar `AH_SOURCE_ZIP` o una ruta al ejecutar.


In [ ]:
from pathlib import Path
import os, sys, json, contextlib, io
import pandas as pd
root = Path.cwd()
if root.name == "notebooks": root = root.parent
if not (root / "src" / "perfilado.ipynb").exists():
    raise RuntimeError("Abrir desde la raíz AH_Adopcion_Digital o su carpeta notebooks")
if not os.environ.get("AH_SOURCE_ZIP"):
    local = root / "input" / "Tablas_Dimensiones_y_Ventas.zip"
    ruta = str(local) if local.is_file() else input("Ruta local del ZIP fuente autorizado: ").strip().strip('"')
    if not Path(ruta).is_file():
        raise FileNotFoundError("Seleccione un ZIP fuente existente")
    os.environ["AH_SOURCE_ZIP"] = ruta
cuaderno_perfilado = root / "src" / "perfilado.ipynb"
%run "$cuaderno_perfilado"
with contextlib.redirect_stdout(io.StringIO()):
    perfil = perfilar_fuentes(os.environ["AH_SOURCE_ZIP"])
print("Perfiladas las siete tablas. Resultados agregados guardados en evidencia/.")

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    cuaderno_diagnosticos = root / "src" / "diagnosticos.ipynb"
    %run "$cuaderno_diagnosticos"
    diagnosticar_fuentes(os.environ["AH_SOURCE_ZIP"])
diag = json.loads((root / "evidencia" / "diagnosticos_adicionales.json").read_text(encoding="utf-8"))
print("Comprobadas las reglas de fecha, importes, clientes, geografía y sensibilidad.")

## 1. Volumen, granularidad y variables

Una fila de ventas es una línea ticket–SKU en estas fuentes. Cuenta y cliente no son equivalentes. Las claves permanecen como texto para no perder ceros o precisión.

In [ ]:
resumen = pd.DataFrame([{ "tabla": k, "filas": v["rows"], "columnas": v["columns"], "duplicados_exactos": v["exact_duplicate_rows"] } for k,v in perfil["tables"].items()])
print(resumen.to_string(index=False))
campos = pd.DataFrame([{ "tabla": k, "campo": c, "tipo": f["dtype"], "nulos": f["nulls"], "distintos": f["distinct_nonnull"] } for k,v in perfil["tables"].items() for c,f in v["fields"].items()])
print("\nINVENTARIO COMPLETO\n" + campos.to_string(index=False))

## 2. Calidad de relaciones

Se comprueba unicidad antes de unir. Un registro sin correspondencia se mantiene visible, sin adjudicar un canal por defecto.

In [ ]:
relaciones = pd.DataFrame([{ "relacion": k, "filas_izquierda": v["left_rows"], "sin_match_no_nulo": v["unmatched_nonnull_rows"], "nulos_clave": v["left_null"], "multiples_matches": v["rows_with_multiple_matches"] } for k,v in perfil["joins"].items()])
print(relaciones.to_string(index=False))

## 3. Distribuciones y cobertura temporal

Las fechas del archivo son ISO, aunque el enunciado describe otro formato. Se usa `%Y-%m-%d`. Los días sin registros no se convierten automáticamente en días con venta cero.

In [ ]:
print("PERIODO", perfil["transactions"]["date_min"], "a", perfil["transactions"]["date_max"])
print("CALENDARIO", json.dumps(diag["dates"], ensure_ascii=False))
print(pd.Series(perfil["transactions"]["rows_by_month"], name="lineas_por_mes").to_string())
for tabla,col in [("FAC_VENTAS","MONTO"),("FAC_VENTAS","ITEMS"),("DIM_CLIENTES","EDAD"),("DIM_MUNICIPIO","IM_2020")]:
    print(tabla, col, json.dumps(perfil["tables"][tabla]["fields"][col], ensure_ascii=False))

## 4. Enriquecimiento territorial y comercial

Las claves municipales permiten unir nombres estatales y, cuando se suministre el archivo oficial ITER, contexto de población y viviendas. Los campos faltantes permanecen nulos. Las claves de producto, tienda y cliente permiten derivar categorías, canal y cohortes con reglas explícitas. Véanse `docs/arquitectura-y-modelo.md` y `docs/matriz-variable-decision.md`.


In [ ]:
print("CLIENTES CON MOVIMIENTOS", json.dumps(diag["active_customers"], ensure_ascii=False))
print("GEOGRAFIA", json.dumps(diag["geography"], ensure_ascii=False))
print("MARCAS", json.dumps(diag["brands"], ensure_ascii=False))
print("SEGMENTOS OBSERVADOS", perfil["tables"]["DIM_SEGMENTOS"]["fields"]["SEGMENTO"]["values"])

## 5. Sensibilidad analítica

Los siguientes escenarios prueban cuánto cambia el universo al modificar la regla. Ninguno demuestra por sí mismo que los negativos son devoluciones ni que excluir servicios sea la interpretación correcta. `ever_online` no equivale a digitalización posterior a una primera compra física.

In [ ]:
print("MONTOS", json.dumps({k:v for k,v in diag["negative_amounts"].items() if k != "by_category"},ensure_ascii=False,indent=2))
print(pd.DataFrame(diag["negative_amounts"]["by_category"]).to_string(index=False))
print(pd.DataFrame.from_dict(diag["amount_rule_sensitivity"],orient="index").to_string())

## 6. Comprobaciones ejecutadas

Estos controles verifican la estructura y la coherencia de los diagnósticos del archivo analizado. No validan Power BI ni resuelven la semántica comercial.

In [ ]:
assert len(perfil["tables"]) == 7
assert perfil["transactions"]["invalid_dates"] == 0
assert perfil["transactions"]["ticket_sku_duplicate_rows_excess"] == 0
assert all(v["rows_with_multiple_matches"] == 0 for v in perfil["joins"].values())
assert diag["dates"]["days_observed"] + diag["dates"]["days_without_rows"] == diag["dates"]["calendar_days"]
assert diag["active_customers"]["age_missing"] <= diag["active_customers"]["customers"]
assert diag["geography"]["state_prefix_matches"] == perfil["tables"]["DIM_MUNICIPIO"]["rows"]
print("Siete controles de estructura y coherencia aprobados sobre el archivo analizado.")

## Gráficos de exploración

Distribución temporal, montos negativos y cobertura de atributos. Cada gráfico usa los agregados del perfilado; las figuras se guardan en evidencia/graficos/.

In [ ]:
cuaderno_graficos = root / "src" / "graficos.ipynb"
%run "$cuaderno_graficos"
from IPython.display import display, Image
for figura in exploratory():
    display(Image(filename=str(figura)))

## Decisiones de la exploración

Se conservan fechas ISO y claves de texto, se agregan cuentas por cliente y no se asigna físico a las tiendas desconocidas. Se mantiene una variante base y tres sensibilidades para importes/servicios. Los catálogos estatales verificados se añaden en el ETL. La población/conectividad municipal se deja nula si no hay ITER local: no se simula una descarga. Las reglas y KPI están en docs/.